# 01 Data Audit & SKU Validation
This notebook inspects sample GPU dataset CSVs, checks schema completeness, missing values, and validates strict separation between 8GB and 16GB RTX 5060 Ti variants.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Add src module to python path
workspace_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(workspace_dir) not in sys.path:
    sys.path.insert(0, str(workspace_dir))

from src.io_utils import load_sample_dataset, load_yaml_config
from src.validate_skus import validate_sku_catalog, validate_listing_mappings
from src.clean_prices import clean_price_observations

## 1. Load Configurations & Products Catalog

In [2]:
products_df = load_sample_dataset("products.csv")
retailers_df = load_sample_dataset("retailers.csv")
listings_df = load_sample_dataset("product_listings.csv")
obs_df = load_sample_dataset("daily_price_observations.csv")
signals_df = load_sample_dataset("external_market_signals.csv")

print("=== Products Catalog ===")
display(products_df.head())

=== Products Catalog ===


,sku_id,gpu_series,chipset,brand,model_name,vram_gb,vram_type,msrp_inr,is_reference_spec
0,RTX5060TI-16G-GIGA-GAMING,RTX 50 Series,RTX 5060 Ti,Gigabyte,GeForce RTX 5060 Ti Gaming OC 16GB,16,GDDR7,49999,False
1,RTX5060TI-8G-MSI-VENTUS,RTX 50 Series,RTX 5060 Ti,MSI,GeForce RTX 5060 Ti Ventus 2X 8GB,8,GDDR7,41999,False
2,RTX5070-12G-ASUS-TUF,RTX 50 Series,RTX 5070,ASUS,TUF Gaming GeForce RTX 5070 12GB,12,GDDR7,64999,True
3,RTX5070TI-16G-ZOTAC-TRIN,RTX 50 Series,RTX 5070 Ti,ZOTAC,Gaming GeForce RTX 5070 Ti Trinity 16GB,16,GDDR7,84999,False
4,RTX5060TI-16G-ASUS-DUAL,RTX 50 Series,RTX 5060 Ti,ASUS,ASUS Dual GeForce RTX 5060 Ti 16GB GDDR7 OC Ed...,16,GDDR7,49999,False


## 2. Validate SKU Catalog & VRAM Separation
We enforce strict separation between 8GB and 16GB products (e.g. RTX 5060 Ti 8GB vs 16GB).

In [3]:
is_cat_valid, cat_errors = validate_sku_catalog(products_df)
print(f"Catalog Valid: {is_cat_valid}")
if cat_errors:
    print("Errors:", cat_errors)
else:
    print("All SKUs pass VRAM separation & consistency checks.")

is_map_valid, map_errors = validate_listing_mappings(listings_df, products_df)
print(f"Listing Mappings Valid: {is_map_valid}")

Catalog Valid: True
All SKUs pass VRAM separation & consistency checks.
Listing Mappings Valid: True


## 3. Audit Price Observations & Local Quotes

In [4]:
cleaned_obs = clean_price_observations(obs_df)
print(f"Raw Observations Count: {len(obs_df)}")
print(f"Cleaned Observations Count: {len(cleaned_obs)}")

local_quotes = cleaned_obs[cleaned_obs["is_local_quote"] == True]
print(f"Local Vendor Quote Observations: {len(local_quotes)}")
display(local_quotes.head())

Raw Observations Count: 169
Cleaned Observations Count: 169
Local Vendor Quote Observations: 26


,observation_id,listing_id,date,price_inr,in_stock,shipping_cost_inr,is_local_quote
26,OBS-1013,LST-5060TI-16G-EAGLE-LOCAL,2026-06-01,59200,True,500,True
27,OBS-1026,LST-5060TI-16G-EAGLE-LOCAL,2026-06-05,60900,True,500,True
28,OBS-1039,LST-5060TI-16G-EAGLE-LOCAL,2026-06-10,63200,True,500,True
29,OBS-1052,LST-5060TI-16G-EAGLE-LOCAL,2026-06-15,65900,True,500,True
30,OBS-1065,LST-5060TI-16G-EAGLE-LOCAL,2026-06-20,68800,True,500,True
